In [2]:
import com.microsoft.spark.fabric
from com.microsoft.spark.fabric.Constants import Constants

df_dwh = spark.read.synapsesql("DWH_Silver_Gold.Silver.tabla_completa_inventario")

df_dwh.show()





StatementMeta(, 5b7cbdab-6ec3-4089-905e-28859aa3fe81, 4, Finished, Available, Finished)

+-----------+------------+-----------------------+--------------+----------------------+------------+--------+-----------+-------+-------------------+---------------+-------------------+-------------+----------------+---------------+-----------------+------------+
|  Categoria|      Estado|Fecha_Ultima_Reposición|          Pais|Cantidad_Minima_Pedido|ID_Proveedor|Longitud|ID_producto|Latitud|      Fecha_Entrada|Precio_Unitario|Cantidad_Inventario|Punto_Reorden|Nivel_Inventario|Nombre_Producto|Ubicacion_Almacen|Días_Entrega|
+-----------+------------+-----------------------+--------------+----------------------+------------+--------+-----------+-------+-------------------+---------------+-------------------+-------------+----------------+---------------+-----------------+------------+
|     Sports|Out of Stock|    2025-09-07 00:00:00|       Germany|                    28|      SUP013| 10.9666|  SKU003431| 51.182|2025-03-18 00:00:00|         250.52|                164|           54|     

In [3]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

df_dim_producto = df_dwh.select("Nombre_Producto", "Categoria").distinct()

window_spec = Window.orderBy("Nombre_Producto", "Categoria")

df_dim_producto = df_dim_producto.withColumn("ID_Producto", row_number().over(window_spec))

df_dim_producto = df_dim_producto.select("ID_Producto", "Nombre_Producto", "Categoria")

df_dim_producto.show()


StatementMeta(, 5b7cbdab-6ec3-4089-905e-28859aa3fe81, 5, Finished, Available, Finished)

+-----------+---------------+---------------+
|ID_Producto|Nombre_Producto|      Categoria|
+-----------+---------------+---------------+
|          1|      Product_1|  Home & Garden|
|          2|     Product_10|           Toys|
|          3|    Product_100|Office Supplies|
|          4|   Product_1000|         Sports|
|          5|   Product_1001|Office Supplies|
|          6|   Product_1002|       Clothing|
|          7|   Product_1003|Office Supplies|
|          8|   Product_1004|  Home & Garden|
|          9|   Product_1005|  Home & Garden|
|         10|   Product_1006|    Electronics|
|         11|   Product_1007|Office Supplies|
|         12|   Product_1008|    Electronics|
|         13|   Product_1009|Office Supplies|
|         14|    Product_101|       Clothing|
|         15|   Product_1010|Office Supplies|
|         16|   Product_1011|    Electronics|
|         17|   Product_1012|  Home & Garden|
|         18|   Product_1013|    Electronics|
|         19|   Product_1014|  Hom

In [4]:
df_dim_producto.write \
    .mode("overwrite") \
    .synapsesql("DWH_Silver_Gold.Gold.dim_producto")



StatementMeta(, 5b7cbdab-6ec3-4089-905e-28859aa3fe81, 6, Finished, Available, Finished)

In [5]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

df_dim_nivel = df_dwh.select("Nivel_Inventario").distinct()
df_dim_nivel = df_dim_nivel.withColumn("ID_Nivel_Inventario", row_number().over(Window.orderBy("Nivel_Inventario")))
df_dim_nivel = df_dim_nivel.select("ID_Nivel_Inventario", "Nivel_Inventario")

df_dim_nivel.write.mode("overwrite").synapsesql("DWH_Silver_Gold.Gold.dim_nivel_inventario")


StatementMeta(, 5b7cbdab-6ec3-4089-905e-28859aa3fe81, 7, Finished, Available, Finished)

In [6]:
df_dim_proveedor = df_dwh.select("ID_Proveedor", "Ubicacion_Almacen").distinct()
df_dim_proveedor = df_dim_proveedor.withColumn("ID_Proveedor_Almacen", row_number().over(Window.orderBy("ID_Proveedor", "Ubicacion_Almacen")))
df_dim_proveedor = df_dim_proveedor.select("ID_Proveedor_Almacen", "ID_Proveedor", "Ubicacion_Almacen")

df_dim_proveedor.write.mode("overwrite").synapsesql("DWH_Silver_Gold.Gold.dim_proveedor_almacen")


StatementMeta(, 5b7cbdab-6ec3-4089-905e-28859aa3fe81, 8, Finished, Available, Finished)

In [7]:
df_dim_estado = df_dwh.select("Estado").distinct()
df_dim_estado = df_dim_estado.withColumn("ID_Estado", row_number().over(Window.orderBy("Estado")))
df_dim_estado = df_dim_estado.select("ID_Estado", "Estado")

df_dim_estado.write.mode("overwrite").synapsesql("DWH_Silver_Gold.Gold.dim_estado")


StatementMeta(, 5b7cbdab-6ec3-4089-905e-28859aa3fe81, 9, Finished, Available, Finished)

In [10]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number
df_dim_ubicacion = df_dwh.select("Pais", "Latitud", "Longitud").distinct()
df_dim_ubicacion = df_dim_ubicacion.withColumn("ID_Ubicacion", row_number().over(Window.orderBy("Pais", "Latitud", "Longitud")))
df_dim_ubicacion = df_dim_ubicacion.select("ID_Ubicacion", "Pais", "Latitud", "Longitud")

df_dim_ubicacion.write.mode("overwrite").synapsesql("DWH_Silver_Gold.Gold.dim_ubicacion")


StatementMeta(, 5b7cbdab-6ec3-4089-905e-28859aa3fe81, 12, Finished, Available, Finished)

In [11]:
from pyspark.sql.functions import min, max

df_dwh.select(
    min("Fecha_Ultima_Reposición").alias("min_fecha_reposicion"),
    max("Fecha_Ultima_Reposición").alias("max_fecha_reposicion"),
    min("Fecha_Entrada").alias("min_fecha_entrada"),
    max("Fecha_Entrada").alias("max_fecha_entrada")
).show()


StatementMeta(, 5b7cbdab-6ec3-4089-905e-28859aa3fe81, 13, Finished, Available, Finished)

+--------------------+--------------------+-------------------+-------------------+
|min_fecha_reposicion|max_fecha_reposicion|  min_fecha_entrada|  max_fecha_entrada|
+--------------------+--------------------+-------------------+-------------------+
| 2023-04-03 00:00:00| 2026-03-05 00:00:00|2023-03-20 00:00:00|2025-03-18 00:00:00|
+--------------------+--------------------+-------------------+-------------------+



In [13]:
from pyspark.sql.functions import col, expr, date_format, dayofweek, year, month, dayofmonth
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number
from datetime import datetime, timedelta

start_date = datetime.strptime("2023-03-20", "%Y-%m-%d")
end_date = datetime.strptime("2026-03-05", "%Y-%m-%d")

fecha_list = [(start_date + timedelta(days=i),) for i in range((end_date - start_date).days + 1)]

df_fechas = spark.createDataFrame(fecha_list, ["Fecha"])

df_fechas = df_fechas.withColumn("ID_Fecha", row_number().over(Window.orderBy("Fecha"))) \
    .withColumn("Año", year("Fecha")) \
    .withColumn("Mes", month("Fecha")) \
    .withColumn("Día", dayofmonth("Fecha")) \
    .withColumn("Nombre_Día", date_format("Fecha", "EEEE")) \
    .withColumn("Nombre_Mes", date_format("Fecha", "MMMM")) \
    .withColumn("Es_Fin_de_Semana", expr("CASE WHEN dayofweek(Fecha) IN (1, 7) THEN 'Sí' ELSE 'No' END"))

df_fechas = df_fechas.select("ID_Fecha", "Fecha", "Año", "Mes", "Día", "Nombre_Día", "Nombre_Mes", "Es_Fin_de_Semana")

df_fechas.show(5)


StatementMeta(, 5b7cbdab-6ec3-4089-905e-28859aa3fe81, 15, Finished, Available, Finished)

+--------+-------------------+----+---+---+----------+----------+----------------+
|ID_Fecha|              Fecha| Año|Mes|Día|Nombre_Día|Nombre_Mes|Es_Fin_de_Semana|
+--------+-------------------+----+---+---+----------+----------+----------------+
|       1|2023-03-20 00:00:00|2023|  3| 20|    Monday|     March|              No|
|       2|2023-03-21 00:00:00|2023|  3| 21|   Tuesday|     March|              No|
|       3|2023-03-22 00:00:00|2023|  3| 22| Wednesday|     March|              No|
|       4|2023-03-23 00:00:00|2023|  3| 23|  Thursday|     March|              No|
|       5|2023-03-24 00:00:00|2023|  3| 24|    Friday|     March|              No|
+--------+-------------------+----+---+---+----------+----------+----------------+
only showing top 5 rows



In [14]:
df_fechas.write.mode("overwrite").synapsesql("DWH_Silver_Gold.Gold.dim_fecha")

StatementMeta(, 5b7cbdab-6ec3-4089-905e-28859aa3fe81, 16, Finished, Available, Finished)

In [16]:
from pyspark.sql.functions import to_date

# 1. Leer la tabla base
df_dwh = spark.read.synapsesql("DWH_Silver_Gold.Silver.tabla_completa_inventario")

df_dwh = df_dwh.withColumn("Fecha_Entrada", to_date("Fecha_Entrada")) \
               .withColumn("Fecha_Ultima_Reposición", to_date("Fecha_Ultima_Reposición"))

# 2. Leer dimensiones
dim_producto   = spark.read.synapsesql("DWH_Silver_Gold.Gold.dim_producto") \
    .withColumnRenamed("ID_Producto", "FK_ID_Producto")

dim_nivel      = spark.read.synapsesql("DWH_Silver_Gold.Gold.dim_nivel_inventario") \
    .withColumnRenamed("ID_Nivel_Inventario", "FK_ID_Nivel_Inventario")

dim_proveedor  = spark.read.synapsesql("DWH_Silver_Gold.Gold.dim_proveedor_almacen") \
    .withColumnRenamed("ID_Proveedor_Almacen", "FK_ID_Proveedor_Almacen")

dim_estado     = spark.read.synapsesql("DWH_Silver_Gold.Gold.dim_estado") \
    .withColumnRenamed("ID_Estado", "FK_ID_Estado")

dim_ubicacion  = spark.read.synapsesql("DWH_Silver_Gold.Gold.dim_ubicacion") \
    .withColumnRenamed("ID_Ubicacion", "FK_ID_Ubicacion")

dim_fecha      = spark.read.synapsesql("DWH_Silver_Gold.Gold.dim_fecha") \
    .withColumnRenamed("ID_Fecha", "FK_ID_Fecha")

dim_fecha_2 = dim_fecha.withColumnRenamed("Fecha", "Fecha_Ultima_Reposición") \
                       .withColumnRenamed("FK_ID_Fecha", "FK_ID_Fecha_Reposicion")

# 3. Realizar los joins y evitar ambigüedad
df_fact = df_dwh \
    .join(dim_producto.select("Nombre_Producto", "Categoria", "FK_ID_Producto"),
          on=["Nombre_Producto", "Categoria"], how="left") \
    .join(dim_nivel.select("Nivel_Inventario", "FK_ID_Nivel_Inventario"),
          on="Nivel_Inventario", how="left") \
    .join(dim_proveedor.select("ID_Proveedor", "Ubicacion_Almacen", "FK_ID_Proveedor_Almacen"),
          on=["ID_Proveedor", "Ubicacion_Almacen"], how="left") \
    .join(dim_estado.select("Estado", "FK_ID_Estado"),
          on="Estado", how="left") \
    .join(dim_ubicacion.select("Pais", "Latitud", "Longitud", "FK_ID_Ubicacion"),
          on=["Pais", "Latitud", "Longitud"], how="left") \
    .join(dim_fecha.selectExpr("Fecha as Fecha_Entrada", "FK_ID_Fecha as FK_ID_Fecha_Entrada"),
          on="Fecha_Entrada", how="left") \
    .join(dim_fecha_2, on="Fecha_Ultima_Reposición", how="left")

# 4. Seleccionar solo columnas finales sin conflicto
df_fact_final = df_fact.select(
    "FK_ID_Producto",
    "FK_ID_Nivel_Inventario",
    "FK_ID_Proveedor_Almacen",
    "FK_ID_Estado",
    "FK_ID_Ubicacion",
    "FK_ID_Fecha_Entrada",
    "FK_ID_Fecha_Reposicion",
    "Cantidad_Inventario",
    "Precio_Unitario",
    "Punto_Reorden",
    "Días_Entrega",
    "Cantidad_Minima_Pedido"
)

# 5. Guardar la tabla final
df_fact_final.write.mode("overwrite").synapsesql("DWH_Silver_Gold.Gold.fact_inventario")


StatementMeta(, 5b7cbdab-6ec3-4089-905e-28859aa3fe81, 18, Finished, Available, Finished)